In [1]:
#Importing packages and setting parameters for initial analysis

import os
import geopandas as gpd
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.append('../src')
import importlib
import ntl_functions

importlib.reload(ntl_functions)
from ntl_functions import plot_NASA_NTL, filter_dataset_by_bounding_box, mask_dataset_by_geometry, yearly_radiance, yearly_nonzero_pixels

In [9]:
SL_berbera = xr.open_dataset("../data/NTL_Data/141225_SL_Berbera_VNP46A4_EY_2012_24.nc")
somaliland_shp_berbera = gpd.read_file("../data/Combined_Datasets/Somaliland_with_berbera/somaliland_with_berbera.shp")

YEARS = range(2012, 2024)
VAR = "NearNadir_Composite_Snow_Free"


#Cities
HGA = gpd.read_file("../data/Combined_Datasets/Hargeisa_shp/HGA.shp")
Berbera = gpd.read_file("../data/Combined_Datasets/Berbera_shp/Berbera.shp")
Burao = gpd.read_file("../data/Combined_Datasets/Burao_shp/Burao.shp")
Boroma = gpd.read_file("../data/Combined_Datasets/Boroma_shp/Boroma.shp")
Erigavo = gpd.read_file("../data/Combined_Datasets/Erigavo_shp/Erigavo.shp")
Las_Anod = gpd.read_file("../data/Combined_Datasets/Las_Anod_shp/Las_Anod.shp")

cities_names = ["Hargeisa",
                "Burao", 
                "Boroma", 
                "Las Anod", 
                "Erigavo",
                "Berbera"
                ]


# For graphing
HGA_NTL = filter_dataset_by_bounding_box(SL_berbera, HGA)
Berbera_NTL = filter_dataset_by_bounding_box(SL_berbera, Berbera)
Burao_NTL = filter_dataset_by_bounding_box(SL_berbera, Burao)
Boroma_NTL = filter_dataset_by_bounding_box(SL_berbera, Boroma)
Erigavo_NTL = filter_dataset_by_bounding_box(SL_berbera, Erigavo)
Las_Anod_NTL = filter_dataset_by_bounding_box(SL_berbera, Las_Anod)

cities_datasets = [HGA_NTL, 
        Burao_NTL,          
        Boroma_NTL,
        Las_Anod_NTL, 
        Erigavo_NTL,
        Berbera_NTL          
         ]

In [10]:
#Showing the total number of pixels in Somaliland, the number of pixels with light detected each year, and the total radiance each year. 
total_rad_and_pixels = pd.DataFrame({
    "Year": list(YEARS),
    "non_zero_pixels": yearly_nonzero_pixels(SL_berbera, YEARS),
    "radiance_data": yearly_radiance(SL_berbera, YEARS)
})

logged_total_rad = pd.DataFrame({
    "Year": list(YEARS),
    "logged_radiance": np.log(total_rad_and_pixels["radiance_data"])
})

regions = {"Maroodi Jeex":[], "Togdheer":[], "Awdal":[], "Sool":[], "Sanaag":[], "Sahil":[]}

for region in regions:
    regions[region] = filter_dataset_by_bounding_box(mask_dataset_by_geometry(SL_berbera, somaliland_shp_berbera[somaliland_shp_berbera["admin1Name"]==region]), somaliland_shp_berbera[somaliland_shp_berbera["admin1Name"]== region])


region_rad_df = pd.DataFrame({
    region: yearly_radiance(region_data, YEARS)
    for region, region_data in regions.items()
})
region_rad_df["Total"] = region_rad_df.sum(axis=1)
region_rad_df.insert(0, "Year", YEARS)


region_pixel_df = pd.DataFrame({
    region: yearly_nonzero_pixels(region_data, YEARS)
    for region, region_data in regions.items()
})
region_pixel_df["Total"] = region_pixel_df.sum(axis=1)
region_pixel_df.insert(0, "Year", YEARS)


cities_rad = pd.DataFrame({
    name: yearly_radiance(cities_datasets, YEARS)
    for name, cities_datasets in zip(cities_names, cities_datasets)
})
cities_rad["Total Cities"] = cities_rad[cities_names].sum(axis=1)
cities_rad.insert(0, "Year", YEARS)


cities_pixel_df = pd.DataFrame({
    name: yearly_nonzero_pixels(cities_datasets, YEARS)
    for name, cities_datasets in zip(cities_names, cities_datasets)
})
cities_pixel_df["Total Cities"] = cities_pixel_df[cities_names].sum(axis=1)
cities_pixel_df.insert(0, "Year", YEARS)


In [11]:
metadata = pd.DataFrame({
    "Item": [
        "Dataset",
        "Variable",
        "Pixel threshold",
        "Years covered",
        "Notes"
    ],
    "Description": [
        "NASA VIIRS VNP46A4",
        "NearNadir_Composite_Snow_Free",
        "Radiance > 0 for pixel counts",
        "2012–2023",
        "Data downloaded on the 14.12.25 using NASA BlackMarble."
    ]
})

with pd.ExcelWriter("../output/ntl_summary_tables.xlsx") as writer:
    metadata.to_excel(writer, sheet_name="README", index=False)
    total_rad_and_pixels.to_excel(writer, sheet_name="Somaliland_Total", index=False)
    region_rad_df.to_excel(writer, sheet_name="Region_Radiance", index=False)
    region_pixel_df.to_excel(writer, sheet_name="Region_Pixels", index=False)
    cities_rad.to_excel(writer, sheet_name="City_Radiance", index=False)
    cities_pixel_df.to_excel(writer, sheet_name="City_Pixels", index=False)
    logged_total_rad.to_excel(writer, sheet_name="Logged_Somaliland_Radiance", index=False)


In [23]:
logged_total_rad.iloc[11,1]-logged_total_rad.iloc[10,1]

np.float64(0.10310968504845341)

In [21]:
print(len(logged_total_rad))

12
